# CNN-EXP-036 — WIDTH64, RF=1029

## Чистая гипотеза

Проверяем **только capacity**.

Reference:

```text
32 channels
8 residual blocks
dilation = 1, 2, 4, 8, 16, 32, 64, 128
RF = 1029 bp
presence + intensity heads
10,000 steps
batch = 4
```

Candidate:

```text
64 channels
8 residual blocks
те же kernel / dilation
тот же RF = 1029 bp
те же targets / loss / sampling / split
те же 10,000 steps
тот же batch = 4
```

**Единственное архитектурное изменение: 32 → 64 channels.**

После обучения не ограничиваемся одной AP:
1. full holdout, memory-safe;
2. overall + каждый contig;
3. TP / random TN / HARD FP score distributions;
4. LOW-TP / HIGH-TP;
5. optional frozen-feature probe: есть ли в 64-D representation сигнал,
   который presence-head ещё не использует.

---

## Перед запуском

Ты хотел изменить `model.py` сам.

Но важно: недостаточно поменять только default внутри `ResidualBlock`.

В factory тоже должны стать 64:

```python
create_cnn_baseline(..., channels: int = 64, ...)
create_cnn_presence_intensity(..., channels: int = 64, ...)
```

Ноутбук ниже **проверит это автоматически** и остановится до обучения,
если реально создаётся 32-channel модель.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA — frozen relative to reference
# ============================================================

split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(
    split.intervals("train")["contig"]
)

validation_contigs = set(
    split.intervals("validation")["contig"]
)

assert train_contigs.isdisjoint(
    validation_contigs
)

display(split.summary)

print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(
    batch_size=4
)

print()
print("Input:", train_batch.x.shape)
print("Both orientations:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Positive position-strand:", int(train_batch.target.sum()))


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3

Input: torch.Size([4, 5, 1284])
Both orientations: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Positive position-strand: 33


## Проверка архитектуры

Ожидаем:

| Stage | Channels | RF |
|---|---:|---:|
| Stem | 5 → 64 | 1 |
| Block 1, k7 d1 | 64 → 64 | 13 |
| Block 2, k3 d2 | 64 → 64 | 21 |
| Block 3, k3 d4 | 64 → 64 | 37 |
| Block 4, k3 d8 | 64 → 64 | 69 |
| Block 5, k3 d16 | 64 → 64 | 133 |
| Block 6, k3 d32 | 64 → 64 | 261 |
| Block 7, k3 d64 | 64 → 64 | 517 |
| Block 8, k3 d128 | 64 → 64 | **1029** |
| Profile head | 64 → 2 | 1029 |

Два выхода головы — это прежние `presence` и `intensity`.
Мы их **не меняем**, чтобы width был единственной переменной.


In [4]:
# ============================================================
# ARCHITECTURE ASSERT
# ============================================================

EXPECTED_CHANNELS = 64
EXPECTED_RF = 1029
EXPECTED_PARAMETER_COUNT = 232_962

architecture_model = create_cnn_presence_intensity()

stem = architecture_model.backbone.stem

actual_channels = int(
    stem.out_channels
)

parameter_count = sum(
    p.numel()
    for p in architecture_model.parameters()
    if p.requires_grad
)

# Compute theoretical RF from actual residual blocks.
rf = 1

for block in architecture_model.backbone.blocks:
    for conv in (block.conv1, block.conv2):
        k = int(conv.kernel_size[0])
        d = int(conv.dilation[0])

        rf += (
            k - 1
        ) * d

print("Stem:", stem.in_channels, "→", stem.out_channels)
print("Residual blocks:", len(architecture_model.backbone.blocks))
print("Theoretical RF:", rf)
print("Profile head:", architecture_model.profile_head.in_channels, "→", architecture_model.profile_head.out_channels)
print("Trainable parameters:", f"{parameter_count:,}")

assert actual_channels == EXPECTED_CHANNELS, (
    f"Сейчас model.py всё ещё создаёт {actual_channels} channels, "
    f"а нужен {EXPECTED_CHANNELS}. "
    "Проверь default channels в create_cnn_baseline() "
    "и create_cnn_presence_intensity()."
)

assert rf == EXPECTED_RF
assert architecture_model.profile_head.in_channels == EXPECTED_CHANNELS
assert architecture_model.profile_head.out_channels == 2
assert parameter_count == EXPECTED_PARAMETER_COUNT

del architecture_model
gc.collect()

print()
print("ARCHITECTURE CHECK: OK")


Stem: 5 → 64
Residual blocks: 8
Theoretical RF: 1029
Profile head: 64 → 2
Trainable parameters: 232,962

ARCHITECTURE CHECK: OK


In [5]:
# ============================================================
# ONE-STEP DIRECTML / GRADIENT SMOKE
# ============================================================

smoke_model = (
    create_cnn_presence_intensity()
    .to(device)
)

smoke_model.train()

smoke_optimizer = DirectMLAdam(
    smoke_model.parameters(),
    lr=1e-3,
)

smoke_batch = train_generator.sample_batch(
    batch_size=4
)

x_gpu = smoke_batch.x_both_strands.to(
    device
)

plus_target_gpu = (
    smoke_batch.plus_target
    .to(device)
)

minus_target_reverse_gpu = (
    smoke_batch.minus_target_reverse
    .to(device)
)

plus_presence_weight_gpu = (
    smoke_batch.plus_loss_weight
    .to(device)
)

minus_presence_weight_reverse_gpu = (
    smoke_batch.minus_loss_weight_reverse
    .to(device)
)

plus_count = (
    smoke_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    smoke_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target_gpu = (
    torch.log1p(plus_count)
    .to(device)
)

minus_intensity_target_reverse_gpu = (
    torch.log1p(minus_count_reverse)
    .to(device)
)

plus_intensity_weight_gpu = (
    2.0
    * smoke_batch.plus_loss_weight
    * (plus_count > 0).float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * smoke_batch.minus_loss_weight_reverse
    * (minus_count_reverse > 0).float()
).to(device)

smoke_optimizer.zero_grad(
    set_to_none=True
)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        smoke_model,
        x_gpu,
        batch_size=4,
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    smoke_loss = (
        presence_loss
        + 0.1 * intensity_loss
    )

    smoke_loss.backward()
    smoke_optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = float(
    smoke_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

intensity_gradient = float(
    smoke_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .detach()
    .cpu()
    .item()
)

print("Presence loss:", float(presence_loss.detach().cpu().item()))
print("Intensity loss:", float(intensity_loss.detach().cpu().item()))
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

del (
    smoke_model,
    smoke_optimizer,
    smoke_batch,
    x_gpu,
    plus_target_gpu,
    minus_target_reverse_gpu,
    plus_presence_weight_gpu,
    minus_presence_weight_reverse_gpu,
    plus_intensity_target_gpu,
    minus_intensity_target_reverse_gpu,
    plus_intensity_weight_gpu,
    minus_intensity_weight_reverse_gpu,
    plus_presence,
    minus_presence_reverse,
    plus_intensity,
    minus_intensity_reverse,
    presence_loss,
    intensity_loss,
    smoke_loss,
)

gc.collect()

print("DIRECTML SMOKE: OK")


Presence loss: 0.689727783203125
Intensity loss: 0.5821141004562378
Presence gradient: 2.957672595977783
Intensity gradient: 1.8705165386199951
CPU fallback: []
DIRECTML SMOKE: OK


## Обучение

Здесь специально оставлены те же:

```text
steps = 10,000
batch = 4
lr = 1e-3
intensity weight = 0.1
seed = 42
```

Это важно: иначе мы уже не сможем сказать, что проверили именно `32 → 64`.


In [6]:
# ============================================================
# EXPERIMENT CONFIG
# ============================================================

EXPERIMENT_ID = "CNN-EXP-036"
EXPERIMENT_SLUG = "CNN_EXP036_WIDTH64_RF1029"

SEED = 42
STEPS = 10_000
BATCH_SIZE = 4
LOG_EVERY = 250

CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}

LEARNING_RATE = 1e-3
INTENSITY_LOSS_WEIGHT = 0.1

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_CHECKPOINT = (
    RUN_DIR
    / "width64_final.pt"
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

print("Experiment:", EXPERIMENT_ID)
print("Artifacts:", RUN_DIR)
print("Channels:", EXPECTED_CHANNELS)
print("RF:", EXPECTED_RF)
print("Steps:", STEPS)
print("Batch:", BATCH_SIZE)


Experiment: CNN-EXP-036
Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001
Channels: 64
RF: 1029
Steps: 10000
Batch: 4


In [7]:
# ============================================================
# TRAINING
# ============================================================

torch.manual_seed(SEED)

generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)

model = (
    create_cnn_presence_intensity()
    .to(device)
)

assert model.backbone.stem.out_channels == 64

optimizer = DirectMLAdam(
    model.parameters(),
    lr=LEARNING_RATE,
)


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return (
            value.detach()
            .cpu()
            .clone()
        )

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def checkpoint_path(step):
    return (
        RUN_DIR
        / f"model_step_{step:05d}.pt"
    )


def save_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": EXPERIMENT_ID,
        "model_type": "presence_intensity",
        "architecture": "rf1029_width64",
        "step": int(step),

        "model_state_dict": model_state_on_cpu(
            model
        ),

        "optimizer_state_dict": tree_to_cpu(
            optimizer.state_dict()
        ),

        "generator_rng_state": copy.deepcopy(
            generator.rng.bit_generator.state
        ),

        "torch_rng_state": torch.get_rng_state(),

        "model_config": {
            "input_channels": 5,
            "channels": 64,
            "output_channels": 2,
            "theoretical_rf": 1029,
        },

        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },

        "training_config": {
            "seed": SEED,
            "steps": STEPS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },

        "seen_position_strand": int(
            seen_position_strand
        ),

        "elapsed_seconds": float(
            elapsed_seconds
        ),

        "history": list(history),
    }

    torch.save(
        checkpoint,
        path,
    )


model.train()

history = []

ema_presence = None
ema_intensity = None
ema_total = None

seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(
        1,
        STEPS + 1,
    ):
        batch = generator.sample_batch(
            batch_size=BATCH_SIZE
        )

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        plus_target_gpu = (
            batch.plus_target
            .to(device)
        )

        minus_target_reverse_gpu = (
            batch.minus_target_reverse
            .to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight
            .to(device)
        )

        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse
            .to(device)
        )

        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )

        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(
                plus_count
            )
            .to(device)
        )

        minus_intensity_target_reverse_gpu = (
            torch.log1p(
                minus_count_reverse
            )
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            model,
            x_gpu,
            batch_size=BATCH_SIZE,
            context_flank=(
                profile_config
                .context_flank
            ),
            target_length=(
                profile_config
                .target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = float(
            presence_loss
            .detach()
            .cpu()
            .item()
        )

        intensity_value = float(
            intensity_loss
            .detach()
            .cpu()
            .item()
        )

        total_value = float(
            total_loss
            .detach()
            .cpu()
            .item()
        )

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )

            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )

            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": int(step),
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": float(
                    ema_presence
                ),
                "ema_intensity": float(
                    ema_intensity
                ),
                "ema_total": float(
                    ema_total
                ),
                "elapsed_seconds": float(
                    elapsed
                ),
                "seen_position_strand": int(
                    seen_position_strand
                ),
                "positions_per_second": float(
                    seen_position_strand
                    / elapsed
                ),
            }

            history.append(
                record
            )

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in CHECKPOINT_STEPS:
            save_checkpoint(
                checkpoint_path(step),
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = (
    time.perf_counter()
    - started_at
)

save_checkpoint(
    FINAL_CHECKPOINT,
    step=STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=history,
)

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print("Total time:", round(elapsed, 2), "sec")
print("Time/step:", round(elapsed / STEPS, 4), "sec")
print("Seen position-strand:", seen_position_strand)
print("Final EMA presence:", round(ema_presence, 6))
print("Final EMA intensity:", round(ema_intensity, 6))
print("Final EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", FINAL_CHECKPOINT)
print("History:", HISTORY_PATH)

assert not fallback_warnings
assert FINAL_CHECKPOINT.is_file()
assert HISTORY_PATH.is_file()


step=    1 presence=0.7496 intensity=0.7713 total=0.8267 ema=(0.7496, 0.7713, 0.8267) positions/s=162,329
step=  250 presence=0.4593 intensity=0.4295 total=0.5022 ema=(0.5566, 0.3566, 0.5923) positions/s=191,348
step=  500 presence=0.5143 intensity=0.0450 total=0.5189 ema=(0.4971, 0.2449, 0.5216) positions/s=193,964
step=  750 presence=0.5590 intensity=0.2845 total=0.5875 ema=(0.4851, 0.2585, 0.5110) positions/s=193,741
step= 1000 presence=0.2922 intensity=0.2110 total=0.3133 ema=(0.4798, 0.2553, 0.5053) positions/s=191,524
step= 1250 presence=0.2551 intensity=0.9154 total=0.3466 ema=(0.4688, 0.2495, 0.4938) positions/s=191,432
step= 1500 presence=0.3565 intensity=0.1357 total=0.3701 ema=(0.4551, 0.2227, 0.4773) positions/s=191,675
step= 1750 presence=0.6579 intensity=0.0802 total=0.6659 ema=(0.4439, 0.2543, 0.4694) positions/s=191,178
step= 2000 presence=0.2895 intensity=0.3052 total=0.3200 ema=(0.4330, 0.2666, 0.4597) positions/s=191,111
step= 2250 presence=0.4626 intensity=0.0836 to

# Сильная validation

Здесь **не используем огромные массивы всех negative positions**.

Full holdout считается streaming:

- presence AP после 5-decimal quantisation — через histogram;
- raw и 5dp EPIC Spearman — только на positive positions;
- intensity Spearman — только на positive positions;
- метрики считаются overall и отдельно по каждому contig;
- одновременно берём небольшие diagnostic samples:
  - random TN;
  - HARD FP candidates (высокий presence score).

Это должно быть существенно безопаснее по RAM.


In [8]:
# ============================================================
# LOAD 10K CHECKPOINT
# ============================================================

checkpoint = torch.load(
    FINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

validation_model = (
    create_cnn_presence_intensity()
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

assert (
    validation_model
    .backbone
    .stem
    .out_channels
    == 64
)

validation_model = validation_model.to(
    device
)

validation_model.eval()

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_tiles))
print(
    "Validation position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)

del checkpoint
gc.collect()


Checkpoint step: 10000
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205


2620

In [9]:
# ============================================================
# METRIC HELPERS
# ============================================================

SUBMISSION_LEVELS = 100_001


def sigmoid_numpy(x):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    out = np.empty_like(
        x,
        dtype=np.float64,
    )

    positive = (
        x >= 0
    )

    out[
        positive
    ] = (
        1.0
        / (
            1.0
            + np.exp(
                -x[
                    positive
                ]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~positive
        ]
    )

    out[
        ~positive
    ] = (
        exp_x
        / (
            1.0
            + exp_x
        )
    )

    return out.astype(
        np.float32
    )


def probability_bin_5dp(
    probabilities,
):
    return np.rint(
        np.clip(
            probabilities,
            0.0,
            1.0,
        )
        * 100_000
    ).astype(
        np.int32
    )


def update_histogram(
    hist,
    indices,
):
    values, counts = np.unique(
        indices,
        return_counts=True,
    )

    hist[
        values
    ] += counts.astype(
        np.int64
    )


def ap_from_histograms(
    positive_hist,
    total_hist,
):
    positives = int(
        positive_hist.sum()
    )

    if positives <= 0:
        return float("nan")

    pos_desc = (
        positive_hist[
            ::-1
        ]
    )

    total_desc = (
        total_hist[
            ::-1
        ]
    )

    tp = np.cumsum(
        pos_desc,
        dtype=np.int64,
    )

    predicted_positive = np.cumsum(
        total_desc,
        dtype=np.int64,
    )

    precision = (
        tp
        / np.maximum(
            predicted_positive,
            1,
        )
    )

    return float(
        np.sum(
            (
                pos_desc
                / positives
            )
            * precision
        )
    )


def dense_rank_pearson(
    prediction,
    target,
):
    prediction = np.asarray(
        prediction
    )

    target = np.asarray(
        target
    )

    if len(prediction) < 2:
        return float("nan")

    pred_rank = rankdata(
        prediction,
        method="dense",
    )

    target_rank = rankdata(
        target,
        method="dense",
    )

    return float(
        pearsonr(
            pred_rank,
            target_rank,
        )[0]
    )


def new_segment_state():
    return {
        "total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_presence_raw": [],
        "positive_presence_5dp": [],
        "positive_intensity": [],
        "positive_count": [],
        "random_negative_score": [],
        "hard_negative_score": [],
    }


In [10]:
# ============================================================
# MEMORY-SAFE FULL HOLDOUT
# ============================================================

EVAL_BATCH_SIZE = 1
PROGRESS_EVERY = 100

RANDOM_NEG_PER_TILE_STRAND = 2
HARD_NEG_PER_TILE_STRAND = 2

rng = np.random.default_rng(
    20260924
)

segment_names = [
    "overall",
    *sorted(
        validation_tiles[
            "contig"
        ].astype(str).unique()
    ),
]

states = {
    segment: new_segment_state()
    for segment in segment_names
}

positions_seen = 0
started_eval = time.perf_counter()

validation_model.eval()

with torch.no_grad():

    for batch_no, start in enumerate(
        range(
            0,
            len(validation_tiles),
            EVAL_BATCH_SIZE,
        ),
        1,
    ):
        selected = (
            validation_tiles
            .iloc[
                start:
                start + EVAL_BATCH_SIZE
            ]
        )

        batch = (
            validation_generator
            .make_batch(
                selected
            )
        )

        B = len(selected)

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        (
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        ) = forward_both_strands_multitask(
            validation_model,
            x_gpu,
            batch_size=B,
            context_flank=(
                profile_config
                .context_flank
            ),
            target_length=(
                profile_config
                .target_length
            ),
        )

        # GPU -> CPU immediately.
        plus_presence_logit = (
            plus_presence_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_presence_logit_reverse = (
            minus_presence_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        plus_intensity = (
            plus_intensity_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        minus_intensity_reverse = (
            minus_intensity_reverse_gpu[
                :,
                0,
                :,
            ]
            .detach()
            .cpu()
            .numpy()
            .astype(
                np.float32,
                copy=True,
            )
        )

        del (
            x_gpu,
            plus_presence_gpu,
            minus_presence_reverse_gpu,
            plus_intensity_gpu,
            minus_intensity_reverse_gpu,
        )

        plus_probability = sigmoid_numpy(
            plus_presence_logit
        )

        minus_probability_reverse = sigmoid_numpy(
            minus_presence_logit_reverse
        )

        plus_count = (
            batch.count[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        minus_count_reverse = (
            batch.count[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                np.float32,
                copy=False,
            )
        )

        plus_mask = (
            batch.mask[
                :,
                0,
                :,
            ]
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        minus_mask_reverse = (
            batch.mask[
                :,
                1,
                :,
            ]
            .flip(-1)
            .numpy()
            .astype(
                bool,
                copy=False,
            )
        )

        for local_i in range(B):
            contig = str(
                selected.iloc[
                    local_i
                ]["contig"]
            )

            for (
                probability,
                intensity,
                count,
                mask,
            ) in (
                (
                    plus_probability[
                        local_i
                    ],
                    plus_intensity[
                        local_i
                    ],
                    plus_count[
                        local_i
                    ],
                    plus_mask[
                        local_i
                    ],
                ),
                (
                    minus_probability_reverse[
                        local_i
                    ],
                    minus_intensity_reverse[
                        local_i
                    ],
                    minus_count_reverse[
                        local_i
                    ],
                    minus_mask_reverse[
                        local_i
                    ],
                ),
            ):
                valid_probability = (
                    probability[
                        mask
                    ]
                )

                valid_intensity = (
                    intensity[
                        mask
                    ]
                )

                valid_count = (
                    count[
                        mask
                    ]
                )

                positive = (
                    valid_count > 0
                )

                score_bin = (
                    probability_bin_5dp(
                        valid_probability
                    )
                )

                for segment in (
                    "overall",
                    contig,
                ):
                    state = states[
                        segment
                    ]

                    update_histogram(
                        state[
                            "total_hist"
                        ],
                        score_bin,
                    )

                    if positive.any():
                        update_histogram(
                            state[
                                "positive_hist"
                            ],
                            score_bin[
                                positive
                            ],
                        )

                        state[
                            "positive_presence_raw"
                        ].append(
                            valid_probability[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_presence_5dp"
                        ].append(
                            (
                                score_bin[
                                    positive
                                ]
                                / 100_000.0
                            ).astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_intensity"
                        ].append(
                            valid_intensity[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        state[
                            "positive_count"
                        ].append(
                            valid_count[
                                positive
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                negative_index = np.flatnonzero(
                    ~positive
                )

                if len(negative_index):

                    n_random = min(
                        RANDOM_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    random_index = rng.choice(
                        negative_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_PER_TILE_STRAND,
                        len(
                            negative_index
                        ),
                    )

                    negative_score = (
                        valid_probability[
                            negative_index
                        ]
                    )

                    if (
                        len(
                            negative_index
                        )
                        <= n_hard
                    ):
                        hard_index = (
                            negative_index
                        )
                    else:
                        local_top = np.argpartition(
                            negative_score,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            negative_index[
                                local_top
                            ]
                        )

                    for segment in (
                        "overall",
                        contig,
                    ):
                        states[
                            segment
                        ][
                            "random_negative_score"
                        ].append(
                            valid_probability[
                                random_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                        states[
                            segment
                        ][
                            "hard_negative_score"
                        ].append(
                            valid_probability[
                                hard_index
                            ].astype(
                                np.float32,
                                copy=True,
                            )
                        )

                positions_seen += int(
                    mask.sum()
                )

        del (
            batch,
            selected,
            plus_presence_logit,
            minus_presence_logit_reverse,
            plus_intensity,
            minus_intensity_reverse,
            plus_probability,
            minus_probability_reverse,
            plus_count,
            minus_count_reverse,
            plus_mask,
            minus_mask_reverse,
        )

        if batch_no % 25 == 0:
            gc.collect()

        if (
            batch_no == 1
            or batch_no % PROGRESS_EVERY == 0
            or start + B >= len(
                validation_tiles
            )
        ):
            elapsed = (
                time.perf_counter()
                - started_eval
            )

            print(
                f"tiles={start + B:>6}"
                f"/{len(validation_tiles)} "
                f"positions={positions_seen:>12,} "
                f"positions/s="
                f"{positions_seen / elapsed:,.0f}"
            )

print()
print("FULL HOLDOUT FINISHED")


tiles=     1/47743 positions=         210 positions/s=19,804
tiles=   100/47743 positions=     125,828 positions/s=147,798
tiles=   200/47743 positions=     269,398 positions/s=161,095
tiles=   300/47743 positions=     403,094 positions/s=163,055
tiles=   400/47743 positions=     552,594 positions/s=169,133
tiles=   500/47743 positions=     708,204 positions/s=173,621
tiles=   600/47743 positions=     871,458 positions/s=177,996
tiles=   700/47743 positions=   1,020,090 positions/s=178,916
tiles=   800/47743 positions=   1,185,336 positions/s=182,377
tiles=   900/47743 positions=   1,346,300 positions/s=184,215
tiles=  1000/47743 positions=   1,519,462 positions/s=187,336
tiles=  1100/47743 positions=   1,700,978 positions/s=190,501
tiles=  1200/47743 positions=   1,853,300 positions/s=190,187
tiles=  1300/47743 positions=   2,017,896 positions/s=191,335
tiles=  1400/47743 positions=   2,193,246 positions/s=192,934
tiles=  1500/47743 positions=   2,381,142 positions/s=195,613
tiles=  1

In [11]:
# ============================================================
# OVERALL + PER-CONTIG METRICS
# ============================================================

metric_rows = []

cohort_cache = {}

for segment, state in states.items():

    positive_raw = np.concatenate(
        state[
            "positive_presence_raw"
        ]
    )

    positive_5dp = np.concatenate(
        state[
            "positive_presence_5dp"
        ]
    )

    positive_intensity = np.concatenate(
        state[
            "positive_intensity"
        ]
    )

    positive_count = np.concatenate(
        state[
            "positive_count"
        ]
    )

    random_negative = np.concatenate(
        state[
            "random_negative_score"
        ]
    )

    hard_negative_candidates = np.concatenate(
        state[
            "hard_negative_score"
        ]
    )

    # Candidate pool contains top negatives from every tile.
    # Keep only its global highest tail.
    hard_keep = min(
        20_000,
        len(
            hard_negative_candidates
        ),
    )

    if (
        len(
            hard_negative_candidates
        )
        > hard_keep
    ):
        hard_idx = np.argpartition(
            hard_negative_candidates,
            -hard_keep,
        )[
            -hard_keep:
        ]

        hard_negative = (
            hard_negative_candidates[
                hard_idx
            ]
        )
    else:
        hard_negative = (
            hard_negative_candidates
        )

    presence_ap_5dp = (
        ap_from_histograms(
            state[
                "positive_hist"
            ],
            state[
                "total_hist"
            ],
        )
    )

    presence_spearman_raw = (
        dense_rank_pearson(
            positive_raw,
            positive_count,
        )
    )

    presence_spearman_5dp = (
        dense_rank_pearson(
            positive_5dp,
            positive_count,
        )
    )

    intensity_spearman = (
        dense_rank_pearson(
            positive_intensity,
            positive_count,
        )
    )

    metric_rows.append({
        "segment": segment,
        "positive_n": int(
            len(
                positive_count
            )
        ),
        "presence_AP_5dp": float(
            presence_ap_5dp
        ),
        "presence_EPIC_Spearman_raw": float(
            presence_spearman_raw
        ),
        "presence_EPIC_Spearman_5dp": float(
            presence_spearman_5dp
        ),
        "intensity_EPIC_Spearman_raw": float(
            intensity_spearman
        ),
    })

    cohort_cache[
        segment
    ] = {
        "positive_score": (
            positive_raw
        ),
        "positive_count": (
            positive_count
        ),
        "random_negative_score": (
            random_negative
        ),
        "hard_negative_score": (
            hard_negative
        ),
    }

metrics = pd.DataFrame(
    metric_rows
).sort_values(
    by="segment"
).reset_index(
    drop=True
)

METRICS_PATH = (
    RUN_DIR
    / "strong_validation_metrics.csv"
)

metrics.to_csv(
    METRICS_PATH,
    index=False,
)

display(metrics)

print("Saved:", METRICS_PATH)


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.033783,0.191207,0.177441,0.203072
1,NC_064041.1,13984,0.028973,0.221014,0.207312,0.234122
2,NC_064042.1,18692,0.038690,0.206500,0.192584,0.220363
3,overall,56205,0.033997,0.177041,0.152906,0.188709


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\strong_validation_metrics.csv


## Ошибки: TP, LOW-TP, HARD-FP, TN

Это не дополнительная метрика соревнования.

Это диагностика вопроса:

> **Что именно изменилось после 32 → 64 channels?**

Определения для overall:

```text
HIGH_TP = верхние 25% score среди positives
LOW_TP  = нижние 25% score среди positives

HARD_FP = глобально самые высоко оценённые negative candidates
RANDOM_TN = случайная выборка negatives
```

Особенно смотрим:

```text
LOW_TP score ↑ ?
HARD_FP score ↓ ?
разрыв TP ↔ random TN вырос?
```


In [12]:
# ============================================================
# ERROR-POPULATION STUDY
# ============================================================

overall = cohort_cache[
    "overall"
]

positive_score = overall[
    "positive_score"
]

positive_count = overall[
    "positive_count"
]

random_tn = overall[
    "random_negative_score"
]

hard_fp = overall[
    "hard_negative_score"
]

q25 = float(
    np.quantile(
        positive_score,
        0.25,
    )
)

q75 = float(
    np.quantile(
        positive_score,
        0.75,
    )
)

low_tp = (
    positive_score[
        positive_score <= q25
    ]
)

high_tp = (
    positive_score[
        positive_score >= q75
    ]
)


def describe_scores(
    name,
    values,
):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    return {
        "cohort": name,
        "n": int(
            len(values)
        ),
        "mean": float(
            values.mean()
        ),
        "p10": float(
            np.quantile(
                values,
                0.10,
            )
        ),
        "p50": float(
            np.quantile(
                values,
                0.50,
            )
        ),
        "p90": float(
            np.quantile(
                values,
                0.90,
            )
        ),
        "p99": float(
            np.quantile(
                values,
                0.99,
            )
        ),
    }


cohort_summary = pd.DataFrame([
    describe_scores(
        "HIGH_TP",
        high_tp,
    ),
    describe_scores(
        "LOW_TP",
        low_tp,
    ),
    describe_scores(
        "ALL_TP",
        positive_score,
    ),
    describe_scores(
        "HARD_FP",
        hard_fp,
    ),
    describe_scores(
        "RANDOM_TN",
        random_tn,
    ),
])

COHORT_PATH = (
    RUN_DIR
    / "error_population_score_summary.csv"
)

cohort_summary.to_csv(
    COHORT_PATH,
    index=False,
)

display(cohort_summary)

print("Positive q25:", q25)
print("Positive q75:", q75)
print("Saved:", COHORT_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,HIGH_TP,14052,0.988612,0.974929,0.990554,0.998835,0.999869
1,LOW_TP,14052,0.311685,0.067290,0.321783,0.537681,0.576590
2,ALL_TP,56205,0.742526,0.261441,0.864286,0.993209,0.999552
3,HARD_FP,20000,0.996571,0.994146,0.996553,0.999030,0.999840
4,RANDOM_TN,190962,0.188906,0.003952,0.072302,0.598810,0.943898


Positive q25: 0.5802421569824219
Positive q75: 0.9693856835365295
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\error_population_score_summary.csv


In [13]:
# ============================================================
# POSITIVE STRENGTH STUDY
# ============================================================

strength_group = np.select(
    [
        positive_count == 1,
        positive_count == 2,
        (
            positive_count >= 3
        )
        & (
            positive_count <= 4
        ),
        positive_count >= 5,
    ],
    [
        "reads_1",
        "reads_2",
        "reads_3_4",
        "reads_5_plus",
    ],
    default="other",
)

strength_rows = []

for group in (
    "reads_1",
    "reads_2",
    "reads_3_4",
    "reads_5_plus",
):
    values = (
        positive_score[
            strength_group == group
        ]
    )

    if len(values) == 0:
        continue

    row = describe_scores(
        group,
        values,
    )

    strength_rows.append(
        row
    )

strength_summary = pd.DataFrame(
    strength_rows
)

STRENGTH_PATH = (
    RUN_DIR
    / "positive_strength_score_summary.csv"
)

strength_summary.to_csv(
    STRENGTH_PATH,
    index=False,
)

display(strength_summary)

print("Saved:", STRENGTH_PATH)


,cohort,n,mean,p10,p50,p90,p99
0,reads_1,29982,0.688090,0.199728,0.791580,0.984455,0.998562
1,reads_2,10496,0.731991,0.250762,0.848369,0.990678,0.999257
2,reads_3_4,6529,0.791000,0.357621,0.908572,0.994611,0.999499
3,reads_5_plus,9198,0.897580,0.679426,0.973530,0.998635,0.999875


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\positive_strength_score_summary.csv


# OPTIONAL — исследуем 64-D representation

Эта часть отвечает уже на более глубокий вопрос:

> presence-head ошибся на HARD FP.  
> Но **есть ли в 64 признаках последнего backbone слоя информация,
> позволяющая отделить эти HARD FP от TP?**

Берём детерминированную подвыборку validation tiles, извлекаем frozen 64-D features
и обучаем диагностический Logistic Regression **cross-contig**.

Это не обучение основной CNN и не влияет на checkpoint.

Интерпретация:

```text
probe TP vs HARD_FP ≈ 0.5
→ backbone сам почти не хранит дополнительного сигнала.

probe заметно > 0.5
→ representation знает больше, чем использует текущая линейная presence-head.
  Тогда следующий эксперимент логично делать с более сильной головой.
```


In [14]:
# ============================================================
# OPTIONAL FEATURE EXTRACTION
# ============================================================

RUN_FEATURE_PROBE = True

DIAG_TILES_PER_CONTIG = 1000
RANDOM_NEG_FEATURES_PER_TILE_STRAND = 2
HARD_NEG_FEATURES_PER_TILE_STRAND = 2

FEATURES_NPZ = (
    RUN_DIR
    / "width64_feature_probe_samples.npz"
)

if RUN_FEATURE_PROBE:

    diag_parts = []

    for contig, frame in (
        validation_tiles
        .groupby(
            "contig",
            sort=True,
        )
    ):
        n = min(
            DIAG_TILES_PER_CONTIG,
            len(frame),
        )

        positions = np.linspace(
            0,
            len(frame) - 1,
            n,
            dtype=int,
        )

        positions = np.unique(
            positions
        )

        diag_parts.append(
            frame.iloc[
                positions
            ]
        )

    diag_tiles = (
        pd.concat(
            diag_parts,
            axis=0,
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Diagnostic tiles:",
        len(diag_tiles),
    )

    rng_probe = np.random.default_rng(
        20260925
    )

    tp_X = []
    tp_score = []
    tp_count = []
    tp_contig = []

    tn_X = []
    tn_score = []
    tn_contig = []

    hard_X = []
    hard_score = []
    hard_contig = []

    validation_model.eval()

    started_probe = time.perf_counter()

    with torch.no_grad():

        for row_i in range(
            len(diag_tiles)
        ):
            selected = (
                diag_tiles
                .iloc[
                    row_i:
                    row_i + 1
                ]
            )

            batch = (
                validation_generator
                .make_batch(
                    selected
                )
            )

            contig = str(
                selected.iloc[
                    0
                ]["contig"]
            )

            x_gpu = (
                batch
                .x_both_strands
                .to(device)
            )

            feature_gpu = (
                validation_model
                .backbone(
                    x_gpu
                )
            )

            output_gpu = (
                validation_model
                .profile_head(
                    feature_gpu
                )
            )

            # CPU first.
            feature = (
                feature_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            output = (
                output_gpu
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32,
                    copy=True,
                )
            )

            del (
                x_gpu,
                feature_gpu,
                output_gpu,
            )

            c = (
                profile_config
                .context_flank
            )

            L = (
                profile_config
                .target_length
            )

            feature = (
                feature[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            output = (
                output[
                    :,
                    :,
                    c:
                    c + L
                ]
            )

            # B=1:
            # index 0 = plus orientation
            # index 1 = reverse-complement orientation.
            plus_count = (
                batch.count[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            minus_count_reverse = (
                batch.count[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            plus_mask = (
                batch.mask[
                    0,
                    0,
                    :,
                ]
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            minus_mask_reverse = (
                batch.mask[
                    0,
                    1,
                    :,
                ]
                .flip(-1)
                .numpy()
                .astype(
                    bool,
                    copy=False,
                )
            )

            for orient_i, (
                count,
                mask,
            ) in enumerate(
                (
                    (
                        plus_count,
                        plus_mask,
                    ),
                    (
                        minus_count_reverse,
                        minus_mask_reverse,
                    ),
                )
            ):
                feat = (
                    feature[
                        orient_i
                    ]
                    .T
                )

                probability = (
                    sigmoid_numpy(
                        output[
                            orient_i,
                            0,
                            :,
                        ]
                    )
                )

                valid_index = np.flatnonzero(
                    mask
                )

                pos_index = valid_index[
                    count[
                        valid_index
                    ] > 0
                ]

                neg_index = valid_index[
                    count[
                        valid_index
                    ] == 0
                ]

                if len(pos_index):
                    tp_X.append(
                        feat[
                            pos_index
                        ].copy()
                    )

                    tp_score.append(
                        probability[
                            pos_index
                        ].copy()
                    )

                    tp_count.append(
                        count[
                            pos_index
                        ].copy()
                    )

                    tp_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            pos_index
                        )
                    )

                if len(neg_index):
                    n_random = min(
                        RANDOM_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    random_index = rng_probe.choice(
                        neg_index,
                        size=n_random,
                        replace=False,
                    )

                    n_hard = min(
                        HARD_NEG_FEATURES_PER_TILE_STRAND,
                        len(
                            neg_index
                        ),
                    )

                    neg_probability = (
                        probability[
                            neg_index
                        ]
                    )

                    if len(
                        neg_index
                    ) <= n_hard:
                        hard_index = (
                            neg_index
                        )
                    else:
                        local_top = np.argpartition(
                            neg_probability,
                            -n_hard,
                        )[
                            -n_hard:
                        ]

                        hard_index = (
                            neg_index[
                                local_top
                            ]
                        )

                    tn_X.append(
                        feat[
                            random_index
                        ].copy()
                    )

                    tn_score.append(
                        probability[
                            random_index
                        ].copy()
                    )

                    tn_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            random_index
                        )
                    )

                    hard_X.append(
                        feat[
                            hard_index
                        ].copy()
                    )

                    hard_score.append(
                        probability[
                            hard_index
                        ].copy()
                    )

                    hard_contig.extend(
                        [
                            contig
                        ]
                        * len(
                            hard_index
                        )
                    )

            del (
                batch,
                selected,
                feature,
                output,
            )

            if (
                row_i == 0
                or (
                    row_i + 1
                ) % 250 == 0
                or row_i + 1
                == len(
                    diag_tiles
                )
            ):
                elapsed = (
                    time.perf_counter()
                    - started_probe
                )

                print(
                    f"probe tiles="
                    f"{row_i + 1:>5}"
                    f"/{len(diag_tiles)} "
                    f"elapsed="
                    f"{elapsed / 60:.1f} min"
                )

            if (
                row_i + 1
            ) % 25 == 0:
                gc.collect()

    tp_X = np.concatenate(
        tp_X,
        axis=0,
    )

    tp_score = np.concatenate(
        tp_score
    )

    tp_count = np.concatenate(
        tp_count
    )

    tp_contig = np.asarray(
        tp_contig
    )

    tn_X = np.concatenate(
        tn_X,
        axis=0,
    )

    tn_score = np.concatenate(
        tn_score
    )

    tn_contig = np.asarray(
        tn_contig
    )

    hard_X = np.concatenate(
        hard_X,
        axis=0,
    )

    hard_score = np.concatenate(
        hard_score
    )

    hard_contig = np.asarray(
        hard_contig
    )

    np.savez_compressed(
        FEATURES_NPZ,
        tp_X=tp_X,
        tp_score=tp_score,
        tp_count=tp_count,
        tp_contig=tp_contig,
        random_tn_X=tn_X,
        random_tn_score=tn_score,
        random_tn_contig=tn_contig,
        hard_fp_X=hard_X,
        hard_fp_score=hard_score,
        hard_fp_contig=hard_contig,
    )

    print()
    print("TP:", tp_X.shape)
    print("Random TN:", tn_X.shape)
    print("Hard FP:", hard_X.shape)
    print("Saved:", FEATURES_NPZ)

else:
    print("Feature probe disabled.")


Diagnostic tiles: 3000
probe tiles=    1/3000 elapsed=0.0 min
probe tiles=  250/3000 elapsed=0.0 min
probe tiles=  500/3000 elapsed=0.1 min
probe tiles=  750/3000 elapsed=0.1 min
probe tiles= 1000/3000 elapsed=0.1 min
probe tiles= 1250/3000 elapsed=0.2 min
probe tiles= 1500/3000 elapsed=0.2 min
probe tiles= 1750/3000 elapsed=0.2 min
probe tiles= 2000/3000 elapsed=0.2 min
probe tiles= 2250/3000 elapsed=0.3 min
probe tiles= 2500/3000 elapsed=0.3 min
probe tiles= 2750/3000 elapsed=0.3 min
probe tiles= 3000/3000 elapsed=0.4 min

TP: (3483, 64)
Random TN: (11998, 64)
Hard FP: (11998, 64)
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\width64_feature_probe_samples.npz


In [15]:
# ============================================================
# OPTIONAL CROSS-CONTIG LINEAR PROBE
# ============================================================

if RUN_FEATURE_PROBE:

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    data = np.load(
        FEATURES_NPZ,
        allow_pickle=False,
    )

    tp_X = data[
        "tp_X"
    ]

    tp_score = data[
        "tp_score"
    ]

    tp_contig = data[
        "tp_contig"
    ]

    random_tn_X = data[
        "random_tn_X"
    ]

    random_tn_score = data[
        "random_tn_score"
    ]

    random_tn_contig = data[
        "random_tn_contig"
    ]

    hard_fp_X = data[
        "hard_fp_X"
    ]

    hard_fp_score = data[
        "hard_fp_score"
    ]

    hard_fp_contig = data[
        "hard_fp_contig"
    ]


    def cross_contig_probe(
        positive_X,
        positive_score,
        positive_contig,
        negative_X,
        negative_score,
        negative_contig,
        task_name,
    ):
        rows = []

        contigs = sorted(
            set(
                positive_contig.tolist()
            )
            & set(
                negative_contig.tolist()
            )
        )

        for heldout in contigs:

            pos_train = (
                positive_contig
                != heldout
            )

            neg_train = (
                negative_contig
                != heldout
            )

            pos_test = (
                positive_contig
                == heldout
            )

            neg_test = (
                negative_contig
                == heldout
            )

            X_train = np.concatenate(
                (
                    positive_X[
                        pos_train
                    ],
                    negative_X[
                        neg_train
                    ],
                ),
                axis=0,
            )

            y_train = np.concatenate(
                (
                    np.ones(
                        pos_train.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_train.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            X_test = np.concatenate(
                (
                    positive_X[
                        pos_test
                    ],
                    negative_X[
                        neg_test
                    ],
                ),
                axis=0,
            )

            y_test = np.concatenate(
                (
                    np.ones(
                        pos_test.sum(),
                        dtype=np.int8,
                    ),
                    np.zeros(
                        neg_test.sum(),
                        dtype=np.int8,
                    ),
                )
            )

            head_score = np.concatenate(
                (
                    positive_score[
                        pos_test
                    ],
                    negative_score[
                        neg_test
                    ],
                )
            )

            probe = make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    solver="lbfgs",
                ),
            )

            probe.fit(
                X_train,
                y_train,
            )

            probe_score = (
                probe.predict_proba(
                    X_test
                )[
                    :,
                    1
                ]
            )

            rows.append({
                "task": task_name,
                "heldout_contig": heldout,
                "n_test": int(
                    len(
                        y_test
                    )
                ),
                "presence_head_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        head_score,
                    )
                ),
                "linear_probe_ROC_AUC": float(
                    roc_auc_score(
                        y_test,
                        probe_score,
                    )
                ),
            })

        return pd.DataFrame(
            rows
        )


    probe_random = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        random_tn_X,
        random_tn_score,
        random_tn_contig,
        "TP_vs_RANDOM_TN",
    )

    probe_hard = cross_contig_probe(
        tp_X,
        tp_score,
        tp_contig,
        hard_fp_X,
        hard_fp_score,
        hard_fp_contig,
        "TP_vs_HARD_FP",
    )

    probe_results = pd.concat(
        (
            probe_random,
            probe_hard,
        ),
        ignore_index=True,
    )

    PROBE_PATH = (
        RUN_DIR
        / "width64_cross_contig_feature_probe.csv"
    )

    probe_results.to_csv(
        PROBE_PATH,
        index=False,
    )

    display(probe_results)

    display(
        probe_results
        .groupby(
            "task",
            as_index=False,
        )[
            [
                "presence_head_ROC_AUC",
                "linear_probe_ROC_AUC",
            ]
        ]
        .mean()
    )

    print("Saved:", PROBE_PATH)

else:
    print("Feature probe disabled.")


,task,heldout_contig,n_test,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_RANDOM_TN,NC_064034.1,5115,0.901960,0.900763
1,TP_vs_RANDOM_TN,NC_064041.1,5035,0.932100,0.931916
2,TP_vs_RANDOM_TN,NC_064042.1,5331,0.916681,0.911287
3,TP_vs_HARD_FP,NC_064034.1,5115,0.309107,0.850381
4,TP_vs_HARD_FP,NC_064041.1,5035,0.390151,0.842538
5,TP_vs_HARD_FP,NC_064042.1,5331,0.327302,0.862490


,task,presence_head_ROC_AUC,linear_probe_ROC_AUC
0,TP_vs_HARD_FP,0.342187,0.851803
1,TP_vs_RANDOM_TN,0.916914,0.914655


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\width64_cross_contig_feature_probe.csv


# Сравнение с WIDTH32 reference

Если старый `CNN_EXP004/run_001/validation_metrics.csv` лежит на месте,
ноутбук просто покажет его рядом.

Мы **не будем подгонять** старые и новые колонки автоматически,
потому что текущая strong-validation пишет более явные названия метрик.

Главные цифры для решения:

```text
WIDTH64 presence_AP_5dp
WIDTH64 presence_EPIC_Spearman_5dp
per-contig stability

и диагностически:
LOW_TP / HARD_FP
TP_vs_HARD_FP feature probe
```


In [16]:
# ============================================================
# REFERENCE DISPLAY
# ============================================================

REFERENCE_METRICS_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP004"
    / "run_001"
    / "validation_metrics.csv"
)

print(
    "WIDTH64 strong validation:"
)

display(
    metrics
)

print()

if REFERENCE_METRICS_PATH.is_file():
    print(
        "WIDTH32 reference:"
    )

    reference_metrics = pd.read_csv(
        REFERENCE_METRICS_PATH
    )

    display(
        reference_metrics
    )

    print(
        "Reference:",
        REFERENCE_METRICS_PATH,
    )

else:
    print(
        "Reference metrics not found:",
        REFERENCE_METRICS_PATH,
    )


WIDTH64 strong validation:


,segment,positive_n,presence_AP_5dp,presence_EPIC_Spearman_raw,presence_EPIC_Spearman_5dp,intensity_EPIC_Spearman_raw
0,NC_064034.1,23529,0.033783,0.191207,0.177441,0.203072
1,NC_064041.1,13984,0.028973,0.221014,0.207312,0.234122
2,NC_064042.1,18692,0.038690,0.206500,0.192584,0.220363
3,overall,56205,0.033997,0.177041,0.152906,0.188709



WIDTH32 reference:


,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,presence_probability,overall,83807486,56205,0.000671,0.027444,40.922419,0.169952,0.143005
1,presence_probability,NC_064034.1,33949114,23529,0.000693,0.025058,36.155206,0.186516,0.171279
2,presence_probability,NC_064041.1,24146754,13984,0.000579,0.025199,43.511483,0.216947,0.200744
3,presence_probability,NC_064042.1,25711618,18692,0.000727,0.033064,45.480372,0.192715,0.176416
4,intensity_prediction,overall,83807486,56205,0.000671,0.026695,39.804316,0.190558,0.198678
5,intensity_prediction,NC_064034.1,33949114,23529,0.000693,0.025145,36.280108,0.204005,0.207033
6,intensity_prediction,NC_064041.1,24146754,13984,0.000579,0.023654,40.843707,0.237998,0.242126
7,intensity_prediction,NC_064042.1,25711618,18692,0.000727,0.031609,43.480085,0.223040,0.227605
8,expected_signal,overall,83807486,56205,0.000671,0.028372,42.304902,0.187643,0.179588
9,expected_signal,NC_064034.1,33949114,23529,0.000693,0.026796,38.662323,0.201736,0.198411


Reference: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP004\run_001\validation_metrics.csv


In [17]:
# ============================================================
# SAVE COMPACT RESULT JSON
# ============================================================

RESULT_JSON = (
    RUN_DIR
    / "EXP036_WIDTH64_RESULT.json"
)

overall_metrics = (
    metrics
    .loc[
        metrics[
            "segment"
        ]
        == "overall"
    ]
    .iloc[
        0
    ]
    .to_dict()
)

result_payload = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "hypothesis": (
            "Doubling backbone width 32→64 improves "
            "TP/TN separation while RF and training protocol stay fixed."
        ),
        "channels": 64,
        "reference_channels": 32,
        "theoretical_rf": 1029,
        "steps": STEPS,
        "batch_size": BATCH_SIZE,
        "only_intended_architecture_change": (
            "backbone width 32→64"
        ),
    },

    "overall": {
        key: (
            int(value)
            if key == "positive_n"
            else float(value)
        )
        for key, value in overall_metrics.items()
        if key != "segment"
    },

    "files": {
        "checkpoint": str(
            FINAL_CHECKPOINT
        ),
        "history": str(
            HISTORY_PATH
        ),
        "strong_validation_metrics": str(
            METRICS_PATH
        ),
        "error_population_summary": str(
            COHORT_PATH
        ),
        "positive_strength_summary": str(
            STRENGTH_PATH
        ),
        "feature_probe_samples": (
            str(
                FEATURES_NPZ
            )
            if RUN_FEATURE_PROBE
            else None
        ),
    },

    "decision_questions": [
        (
            "Did full-holdout AP improve over width32?"
        ),
        (
            "Is the improvement consistent across all validation contigs?"
        ),
        (
            "Did LOW_TP scores rise and/or HARD_FP scores fall?"
        ),
        (
            "Can frozen 64-D features separate TP from HARD_FP "
            "better than the current presence head?"
        ),
        (
            "If width64 clearly improves, run the same protocol at width128."
        ),
    ],
}

RESULT_JSON.write_text(
    json.dumps(
        result_payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("=" * 72)
print("EXP036 COMPLETE")
print("=" * 72)
print()
print("Send me:")
print(RESULT_JSON)


EXP036 COMPLETE

Send me:
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\EXP036_WIDTH64_RESULT.json


In [20]:
# ============================================================
# ADAPTER FOR OLD EXPERIMENT REPORTER
# ============================================================

REPORT_METRICS_PATH = (
    RUN_DIR
    / "validation_metrics_report.csv"
)

report_metrics = (
    metrics[
        [
            "segment",
            "presence_AP_5dp",
            "presence_EPIC_Spearman_raw",
        ]
    ]
    .rename(
        columns={
            "presence_AP_5dp": "average_precision",
            "presence_EPIC_Spearman_raw": "epic_spearman",
        }
    )
)

report_metrics.to_csv(
    REPORT_METRICS_PATH,
    index=False,
)

display(report_metrics)

print("Reporter metrics:", REPORT_METRICS_PATH)

,segment,average_precision,epic_spearman
0,NC_064034.1,0.033783,0.191207
1,NC_064041.1,0.028973,0.221014
2,NC_064042.1,0.038690,0.206500
3,overall,0.033997,0.177041


Reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\validation_metrics_report.csv


In [22]:
CARD_PATH = (
    PROJECT_ROOT
    / "cnn_exp"
    / f"{EXPERIMENT_ID}.md"
)

CARD_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if not CARD_PATH.exists():
    CARD_PATH.write_text(
        f"""---
id: {EXPERIMENT_ID}
type: experiment
experiment_type: cnn-candidate
status: completed
decision: review
date: 2026-09-24
seed: 42
split_version: contig_holdout_v1
primary_metric: average_precision
secondary_metric: epic_spearman
reference_experiment: CNN-EXP-004
tags:
  - ml/experiment
  - ml/cnn
---

# {EXPERIMENT_ID} — WIDTH64 RF1029

← [[cnn_exp/_index.md|Реестр CNN]]

## Pre-registration

- **Гипотеза:** увеличение ширины backbone с 32 до 64 каналов улучшит качество при неизменном RF=1029.
- **Единственное изменение:** channels 32 → 64.
- **Что заморожено:** split, generator, targets, loss, optimizer, RF, validation protocol.
- **Reference:** [[cnn_exp/CNN-EXP-004.md|CNN-EXP-004]].
- **Критерий:** AP выше reference; проверить стабильность по contigs и структуру ошибок.

## Автоматический отчёт

<!-- auto:cnn-experiment-report:start -->

<!-- auto:cnn-experiment-report:end -->

## Ручные заметки

-
""",
        encoding="utf-8",
    )

print("Card:", CARD_PATH)

Card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-036.md


In [23]:
# ============================================================
# OPTIONAL: REGISTER EXPERIMENT AFTER REVIEW
# ============================================================

SAVE_EXPERIMENT = True

FINAL_DECISION = "review"
FINAL_NOTES = (
    "WIDTH64 capacity test. Review strong validation, "
    "error populations and cross-contig feature probe first."
)

if SAVE_EXPERIMENT:

    from ml_project.cnn_experiment_report import (
        finalize_cnn_experiment,
    )

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id=EXPERIMENT_ID,
        title="WIDTH64 RF1029",
        hypothesis=(
            "32→64 channels improves TP/TN separation "
            "with RF1029 and protocol frozen"
        ),
        notebook_path=(
            "notebooks/experiments/CNN_experiments/"
            "CNN_EXP036_WIDTH64_RF1029_STRONG_VALIDATION.ipynb"
        ),
        artifact_dir=RUN_DIR,
        training_history=HISTORY_PATH,
        candidate_metrics=REPORT_METRICS_PATH,
        reference_experiment="CNN-EXP-004",
        reference_metrics=(
            "artifacts/experiments/"
            "CNN_EXP004/run_001/"
            "validation_metrics.csv"
        ),
        decision=FINAL_DECISION,
        notes=FINAL_NOTES,
    )

    print(
        "Obsidian:",
        finalization.card_path,
    )

    print(
        "Summary:",
        finalization.summary_path,
    )

    print(
        "Plots:",
        finalization.plot_dir,
    )

else:
    print(
        "SAVE_EXPERIMENT=False. "
        "Сначала пришли мне EXP036_WIDTH64_RESULT.json."
    )


Obsidian: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-036.md
Summary: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\experiment_summary.json
Plots: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\plots
